In [4]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
raw_path = "Files/bronze/supply_chain/DataCoSupplyChainDataset.csv"

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(raw_path)

print("File Loaded Successfully")

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 6, Finished, Available, Finished, False)

File Loaded Successfully


In [2]:
print(df_raw.count())

StatementMeta(, 57b1f159-7611-4ce5-b5ea-2d1a35ee1ead, 4, Finished, Available, Finished, False)

180519


In [5]:
df_raw.printSchema()

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 7, Finished, Available, Finished, False)

root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer City: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer Fname: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Lname: string (nullable = true)
 |-- Customer Password: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Customer State: string (nullable = true)
 |-- Customer Street: string (nullable = true)
 |-- Customer Zipcode: integer (nullable = true)
 |-- Department Id: integer (nullable = 

In [6]:
display(df_raw.limit(10))

StatementMeta(, 57b1f159-7611-4ce5-b5ea-2d1a35ee1ead, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d4a016b-368d-4579-8550-0b56f6f351de)

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import uuid

pipeline_run_id = str(uuid.uuid4())

print("Pipeline Run ID:")
print(pipeline_run_id)

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 3, Finished, Available, Finished, False)

Pipeline Run ID:
4e8c9fbe-7b7f-4c1b-acdd-89c140942d83


In [6]:
source_file_name = "DataCoSupplyChainDataset.csv"

df_bronze = (
    df_raw
    .withColumn("source_file", lit(source_file_name))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("pipeline_run_id", lit(pipeline_run_id))
    .withColumn("load_date", current_date())
)

display(df_bronze.limit(5))

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4f81d73-c0d0-45ed-8365-fad6a812719a)

In [8]:
#converting colnames to snake_case
import re

def clean_column_name(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r'[^a-zA-Z0-9]', '_', col_name)
    col_name = re.sub(r'_+', '_', col_name)
    return col_name.strip('_')

new_columns = [
    clean_column_name(col)
    for col in df_bronze.columns
]

df_bronze = df_bronze.toDF(*new_columns)

print("Column names standardized")
print(df_bronze.columns[:20])

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 10, Finished, Available, Finished, False)

Column names standardized
['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode']


In [9]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_supply_chain_raw")
)

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 11, Finished, Available, Finished, False)

In [10]:
spark.table("bronze_supply_chain_raw").count()

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 12, Finished, Available, Finished, False)

180519

In [12]:
dict_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/reference/DescriptionDataCoSupplyChain.csv")
)

display(dict_df.limit(5))

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, afdae6b6-1bbd-4526-825c-c7f6b9b6ca2c)

In [13]:
(
    dict_df
    .withColumn("source_file", lit("DescriptionDataCoSupplyChain.csv"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("pipeline_run_id", lit(pipeline_run_id))
    .withColumn("load_date", current_date())
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_data_dictionary_raw")
)

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 15, Finished, Available, Finished, False)

In [14]:
logs_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/logs/tokenized_access_logs.csv")
)

display(logs_df.limit(5))

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b65b5e42-704d-4aed-a73a-037c2db94a54)

In [15]:
(
    logs_df
    .withColumn("source_file", lit("tokenized_access_logs.csv"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("pipeline_run_id", lit(pipeline_run_id))
    .withColumn("load_date", current_date())
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_access_logs_raw")
)

StatementMeta(, 0a2f6ec9-cee5-48b0-b220-d86c63dafaea, 17, Finished, Available, Finished, False)